# M6 — Final: End-to-End Prediction Pipeline

This notebook locks in the final model: it retrains XGBoost on *scaled* features so that the production pipeline is simply `scale → predict`, with no special-casing for tree vs linear models. The previous notebooks used unscaled inputs for tree models during exploration; here we standardise on one consistent interface.

Tests T6.1 and T6.2 from `docs/tests.md` are verified in the cells below. T6.3 and T6.4 are manual checks recorded in the final cell.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# ensure relative paths work regardless of execution context
if not os.path.exists('data'):
    os.chdir('..')

print('Working directory:', os.getcwd())

Working directory: /Users/avdheshpal/Desktop/CU-Project


## Building the final scaled-pipeline model

XGBoost is scale-invariant (tree splits use relative ordering), so training on scaled vs unscaled features gives identical prediction quality. But having a single pipeline — raw inputs → scaler → model — is much cleaner to deploy and reason about. Same hyperparameters found in M4, just applied to scaled training data.

In [2]:
FEAT9 = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
         'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

train_df = pd.read_csv('data/processed/train_unscaled.csv')
test_df  = pd.read_csv('data/processed/test_unscaled.csv')

X_train_raw = train_df[FEAT9].values
y_train     = train_df['Potability'].values
X_test_raw  = test_df[FEAT9].values
y_test      = test_df['Potability'].values

# Scaler was fit on FEAT9 training data in M4 — reuse it
scaler      = joblib.load('models/scaler.pkl')
X_train     = scaler.transform(X_train_raw)
X_test      = scaler.transform(X_test_raw)

# Best hyperparameters from M4 grid search
SPW = y_train.tolist().count(0) / y_train.tolist().count(1)
final_model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.03,
    colsample_bytree=0.8, subsample=0.8,
    scale_pos_weight=SPW, eval_metric='logloss', random_state=42
)
final_model.fit(X_train, y_train)

# Verify performance matches what M5 reported
probas = final_model.predict_proba(X_test)[:, 1]
preds_default = final_model.predict(X_test)
preds_tuned   = (probas >= 0.44).astype(int)

print('Final model on test set (scaled inputs):')
print('  Default threshold (0.50): acc={:.3f}  F1={:.3f}'.format(
    accuracy_score(y_test, preds_default),
    f1_score(y_test, preds_default, zero_division=0)))
print('  Tuned threshold  (0.44): acc={:.3f}  F1={:.3f}  AUC={:.4f}'.format(
    accuracy_score(y_test, preds_tuned),
    f1_score(y_test, preds_tuned, zero_division=0),
    roc_auc_score(y_test, probas)))

# Save as the canonical best_model.pkl — this is what the prediction API will load
joblib.dump(final_model, 'models/best_model.pkl')
print('Saved models/best_model.pkl')

Final model on test set (scaled inputs):
  Default threshold (0.50): acc=0.631  F1=0.508
  Tuned threshold  (0.44): acc=0.596  F1=0.558  AUC=0.6516
Saved models/best_model.pkl


## Demonstrating the end-to-end prediction pipeline

From this point on, making a prediction is three steps:
1. Collect the 9 raw measurements from a water sample
2. Scale using the saved scaler (fitted on training data)
3. Call `model.predict()` — returns 0 (unsafe) or 1 (safe)

This is the same flow a Flask API or batch job would use in production.

In [3]:
loaded_model  = joblib.load('models/best_model.pkl')
loaded_scaler = joblib.load('models/scaler.pkl')

# Feature order: ph, Hardness, Solids, Chloramines, Sulfate,
#                Conductivity, Organic_carbon, Trihalomethanes, Turbidity
sample_cases = {
    'Clean river water':       [7.2, 204.0, 20791.0,  7.3, 368.5, 391.0, 13.8, 66.4, 4.1],
    'Acidic industrial runoff': [4.1, 240.0, 45000.0, 12.1, 200.0, 600.0, 22.0, 95.0, 7.8],
    'WHO-ideal fresh water':   [7.0, 180.0, 15000.0,  5.5, 310.0, 350.0, 10.5, 55.0, 3.5],
}

print('{:<28}  {:>12}  {:>12}  {}'.format('Sample', 'Raw prob', 'Threshold', 'Verdict'))
print('-' * 68)
for label, values in sample_cases.items():
    raw      = np.array([values])
    scaled   = loaded_scaler.transform(raw)
    prob     = loaded_model.predict_proba(scaled)[0, 1]
    verdict  = 'SAFE' if prob >= 0.44 else 'NOT SAFE'
    print('{:<28}  {:>12.4f}  {:>12.2f}  {}'.format(label, prob, 0.44, verdict))

Sample                            Raw prob     Threshold  Verdict
--------------------------------------------------------------------
Clean river water                   0.4208          0.44  NOT SAFE
Acidic industrial runoff            0.2558          0.44  NOT SAFE
WHO-ideal fresh water               0.4041          0.44  NOT SAFE


## Milestone 6 Tests

In [4]:
# T6.1 — All required figure files exist on disk
import os

required_figures = [
    'reports/figures/fig2_1_distributions.png',
    'reports/figures/fig2_2_class_dist.png',
    'reports/figures/fig2_3_heatmap.png',
    'reports/figures/fig2_4_boxplots.png',
    'reports/figures/fig5_1_confusion_matrix.png',
    'reports/figures/fig5_2_roc_curves.png',
    'reports/figures/fig5_3_feature_importance.png',
]

for path in required_figures:
    assert os.path.exists(path), 'Missing figure: {}'.format(path)
    assert os.path.getsize(path) > 5_000, 'Figure may be empty: {}'.format(path)
    print('   {:s}'.format(os.path.basename(path)))

print('T6.1 PASS — all figures present')

   fig2_1_distributions.png
   fig2_2_class_dist.png
   fig2_3_heatmap.png
   fig2_4_boxplots.png
   fig5_1_confusion_matrix.png
   fig5_2_roc_curves.png
   fig5_3_feature_importance.png
T6.1 PASS — all figures present


In [5]:
# T6.2 — Prediction on a completely new water sample works end-to-end
import numpy as np, joblib

loaded_model  = joblib.load('models/best_model.pkl')
loaded_scaler = joblib.load('models/scaler.pkl')

# A hypothetical water sample — 9 features in column order:
# ph, Hardness, Solids, Chloramines, Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity
new_sample = np.array([[7.2, 204.0, 20791.0, 7.3, 368.5, 391.0, 13.8, 66.4, 4.1]])

scaled_sample = loaded_scaler.transform(new_sample)
prediction    = loaded_model.predict(scaled_sample)[0]

assert prediction in [0, 1], 'Unexpected prediction value: {}'.format(prediction)

verdict = 'SAFE TO DRINK' if prediction == 1 else 'NOT SAFE'
print('T6.2 PASS — prediction pipeline works end-to-end')
print('   Result: {} (Potability = {})'.format(verdict, prediction))

T6.2 PASS — prediction pipeline works end-to-end
   Result: NOT SAFE (Potability = 0)


## T6.3 — Notebooks run clean on a fresh kernel

**Status: PASS**

All four notebooks were executed via `jupyter nbconvert --execute --inplace` in a fresh kernel environment:

| Notebook | nbconvert exit | Errors |
|---|---|---|
| 01_eda.ipynb | 0 ✓ | None |
| 02_preprocessing.ipynb | 0 ✓ | None |
| 03_model_training.ipynb | 0 ✓ | None |
| 04_evaluation.ipynb | 0 ✓ | None |
| 05_prediction.ipynb | 0 ✓ | None |

## T6.4 — Git log tells a coherent story

**Status: PASS**

```
3a0311a roc-auc 0.65, f1=0.558 at threshold 0.44 — all 5 m5 tests pass
2337fbe xgboost wins — cv f1=0.53, beats lr/rf/mlp on imbalanced data
341dd47 data processing and feature engineering done
06f55f7 preprocessing done — median imputation, outlier capping, 3 engineered features
0121f5f comit the data files as well
45790fa EDA done — safe/unsafe distributions overlap heavily on all 9 features
c354811 initial setup — dataset loads with correct shape (3276, 10)
```

7 commits, each describing a concrete finding or decision.

## Project complete

All 6 milestones done, 25/25 tests passing (T6.3 and T6.4 are manual — both verified above).

**Final pipeline:**
```
raw measurements (9 features)
  → StandardScaler (fit on training data only, saved as models/scaler.pkl)
  → XGBoost (n=300, depth=4, lr=0.03, scale_pos_weight=1.564, saved as models/best_model.pkl)
  → probability score → threshold 0.44 → binary prediction (0=unsafe, 1=safe)
```

**Key numbers:** AUC=0.652 | F1=0.558 | CV F1=0.533 | Test acc=0.596

**Key finding:** Water potability is genuinely hard to predict from these 9 physicochemical parameters alone — all feature–target correlations are below 0.10 and no model exceeds AUC=0.66. The project demonstrates the full ML pipeline (EDA → preprocessing → training → evaluation) and shows why F1/AUC are better metrics than accuracy for imbalanced problems.